In [1]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from dotenv import load_dotenv
import os

In [2]:
from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
class State(TypedDict):
    topic: str
    advertisement: str
    review: str
    tagline: str
    combined_output: str
    


In [5]:
def generate_advertisement(state: State) -> State:
    """Calls OpenAI API to generate an advertisement related to the given topic."""
    msg = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a creative AI that writes catchy advertisements."},
            {"role": "user", "content": f"Write a catchy advertisement for a product related to {state['topic']}."}
        ],
        max_tokens=200
    )
    advertisement = msg.choices[0].message.content.strip()
    return {"advertisement": advertisement}

In [7]:
# Step 4: Generate a product review
def generate_review(state: State):
    """Calls OpenAI API to generate a detailed product review for the given topic."""
    msg = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that writes detailed product reviews."},
            {"role": "user", "content": f"Write a product review for a product related to {state['topic']}. Include pros and cons."}
        ],
        max_tokens=300
    )
    review = msg.choices[0].message.content.strip()
    return {"review": review}

In [8]:
# Step 5: Generate a catchy tagline
def generate_tagline(state: State):
    """Calls OpenAI API to generate a catchy tagline for the given topic."""
    msg = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a creative AI that generates catchy taglines."},
            {"role": "user", "content": f"Create a short, catchy tagline for a product related to {state['topic']}."}
        ],
        max_tokens=50
    )
    tagline = msg.choices[0].message.content.strip()
    return {"tagline": tagline}


In [9]:
def combine_outputs(state: State):
    """Combines advertisement, review, and tagline into a single output."""
    combined = (
        f"Advertisement:\n{state['advertisement']}\n\n"
        f"Review:\n{state['review']}\n\n"
        f"Tagline:\n{state['tagline']}"
    )
    return {"combined_output": combined}

In [12]:
def build_workflow():
    "construct and compile the langraph parallel workflow"
    parallel_builder = StateGraph(State)
    parallel_builder.add_node("generate_advertisement", generate_advertisement)
    parallel_builder.add_node("generate_review", generate_review)
    parallel_builder.add_node("generate_tagline", generate_tagline)
    parallel_builder.add_node("combine_outputs", combine_outputs)

    parallel_builder.add_edge(START, "generate_advertisement")
    parallel_builder.add_edge(START, "generate_review")
    parallel_builder.add_edge(START, "generate_tagline")

    parallel_builder.add_edge("generate_advertisement", "combine_outputs")
    parallel_builder.add_edge("generate_review", "combine_outputs")
    parallel_builder.add_edge("generate_tagline", "combine_outputs")

    parallel_builder.add_edge("combine_outputs", END)

    workflow = parallel_builder.compile()
    return workflow

    



In [13]:
parallel_workflow = build_workflow()

topic="smart home devices"
state = parallel_workflow.invoke({"topic": topic})

result=parallel_workflow.invoke(state)


In [14]:
result

{'topic': 'smart home devices',
 'advertisement': '🌟 Transform Your Home into a Smart Sanctuary! 🌟\n\nIntroducing the *HomeGenius Smart Hub* – where convenience meets comfort! 🏡✨\n\n🔌 **Connect with Ease**: Control all your devices with a single touch or voice command. Lights, locks, and appliances – your wish is their command!\n\n🔊 **Stay in Tune**: Enjoy a seamless sound experience with our built-in smart speakers. Play your favorite tunes or catch up on the latest news – all from the heart of your home.\n\n🛡️ **Safety First**: Monitor your home in real-time with our advanced security features – alerts, cameras, and 24/7 access, right at your fingertips!\n\n💡 **Energy Smart**: Save on bills and live sustainably with our energy-saving features. Schedule your devices to run only when you need them.\n\n💬 **Your Home, Your Rules**: Personalize your environment with customizable settings. Create the perfect ambiance for any occasion or mood',
 'review': '**Product Review: Philips Hue Whit